# STKI — RAG (Retrieval-Augmented Generation) dengan IndoBERT + GPT

Notebook ini melengkapi metode sebelumnya (TF-IDF & Vector Space Model). Pada notebook
TF-IDF/VSM, sistem **hanya dapat me-ranking dokumen** terhadap sebuah query, tetapi
**tidak dapat menjawab pertanyaan**. Di sini kita menambahkan **Large Language Model (LLM)**
ke dalam pipeline temu kembali sehingga sistem mampu **menjawab pertanyaan dalam bahasa alami**
beserta **referensi dokumen sumber**.

**Alur RAG:**

1. **Korpus** — 10 dokumen hukum pajak (PBB & Pajak Kendaraan Bermotor) dalam format PDF.
2. **Chunking** — setiap PDF dipecah menjadi banyak *passage* berukuran bervariasi.
3. **Embedding** — setiap passage dikodekan menjadi vektor dengan transformer **IndoBERT**.
4. **Retrieval** — query dikodekan dengan IndoBERT lalu dicari passage paling relevan via *cosine similarity* (Dense VSM).
5. **Generation** — passage teratas dikirim ke **GPT (`gpt-4o-mini` via OpenRouter)** sebagai konteks; LLM menyusun jawaban dan mengutip dokumen `[D#]`.

> Korpus tema **Hukum Pajak**: Undang-Undang, Peraturan Menteri Keuangan (PMK),
> Peraturan Menteri Dalam Negeri (Permendagri), dan Modul akademik Universitas Terbuka.


## 1. Install Library

Jalankan sel ini hanya jika library belum terpasang di environment. Pada venv proyek
(`requirements.txt`) library berikut sudah tersedia.


In [1]:
# !pip install torch --index-url https://download.pytorch.org/whl/cpu
# !pip install transformers sentencepiece openai python-dotenv pdfplumber pandas numpy

## 2. Import & Konfigurasi

Kunci API dibaca dari file `.env` (variabel `OPENROUTER_API_KEY`). File `.env` **tidak**
di-commit ke git. Karena kunci berformat `sk-or-v1-...` (OpenRouter), kita memakai SDK
`openai` dengan `base_url` OpenRouter dan model id `openai/gpt-4o-mini`.


In [2]:
import os, re, math
from pathlib import Path

# Cache model HuggingFace disimpan lokal (folder ini di-gitignore)
os.environ.setdefault("HF_HOME", str(Path("hf_cache").resolve()))
# Batasi jumlah thread BLAS sebelum import torch/numpy (cegah CPU 100% semua core)
N_THREAD = "4"
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ.setdefault(_v, N_THREAD)

import numpy as np
import pandas as pd
import pdfplumber
import torch
torch.set_num_threads(int(N_THREAD))   # batasi CPU PyTorch agar mesin tetap responsif
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
from openai import OpenAI

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 50)

# ---- Konstanta ----
KORPUS_DIR        = Path("corpus_pajak")
# IndoBERT (indobenchmark/indobert-base-p1) yang di-fine-tune menjadi Sentence-BERT
# sehingga embedding-nya cocok untuk semantic search / retrieval.
MODEL_EMBED       = "firqaaa/indo-sentence-bert-base"
MODEL_LLM         = "openai/gpt-4o-mini"                # generator (OpenRouter)
TOP_K             = 5                                   # jumlah passage konteks untuk LLM
KALIMAT_PER_CHUNK = 4                                   # ukuran passage (jumlah kalimat)
MAX_KATA_CHUNK    = 160                                 # batas atas kata/passage (limit token model)
MIN_CHARS         = 40                                  # buang passage terlalu pendek
WORDY_MIN         = 0.60                                # buang passage yang didominasi angka/kode (tabel tarif)

# ---- Klien LLM (OpenRouter) ----
load_dotenv(Path(".env"))
_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not _API_KEY:
    raise RuntimeError(
        "OPENROUTER_API_KEY tidak ditemukan. Buat file lsi-colab/.env berisi:\n"
        "OPENROUTER_API_KEY=sk-or-v1-...."
    )
client = OpenAI(api_key=_API_KEY, base_url="https://openrouter.ai/api/v1")
print("Konfigurasi siap. Model embedding:", MODEL_EMBED, "| Model LLM:", MODEL_LLM)

Konfigurasi siap. Model embedding: firqaaa/indo-sentence-bert-base | Model LLM: openai/gpt-4o-mini


## 3. Korpus & Chunking

Sepuluh PDF hukum pajak dipetakan ke ID `D1`–`D10`. Karena dokumen hukum berukuran besar
(puluhan halaman), masing-masing dipecah menjadi banyak **passage**. Inilah unit yang nantinya
di-*retrieve* dan dikutip oleh LLM (mis. `D6#42` = passage ke-42 dari Permendagri 7/2025).

Beberapa dokumen (Permendagri) memuat **tabel tarif** tanpa tanda baca akhir kalimat, sehingga
satu "kalimat" bisa sangat panjang. Karena itu passage dibatasi maksimal `MAX_KATA_CHUNK` kata
agar tidak melampaui batas token model (512 token). Selain itu, passage yang **didominasi angka/kode**
(baris tabel tarif, nomor induk, dsb.) dibuang lewat filter `rasio_kata >= WORDY_MIN` karena tidak
informatif untuk tanya-jawab dan justru menambah noise pada retrieval.


In [3]:
# Pemetaan file PDF -> (ID dokumen, label sumber untuk sitasi)
PETA_DOKUMEN = {
    "85uu012.pdf":                       ("D1",  "UU 12/1985 - PBB"),
    "UU Nomor 12 Tahun 1985.pdf":        ("D2",  "UU 12/1985 - PBB (salinan)"),
    "2024pmkeuangan085.pdf":             ("D3",  "PMK 85/2024 - Penilaian NJOP PBB-P2"),
    "234~PMK.03~2022Per.pdf":            ("D4",  "PMK 234/2022 - PBB pertambangan/kehutanan"),
    "PAJA323304-M1.pdf":                 ("D5",  "Modul PBB Universitas Terbuka"),
    "Permendagri Nomor 7 Tahun 2025.pdf":("D6",  "Permendagri 7/2025 - PKB & BBN-KB 2025"),
    "Permendagri No 8 Tahun 2024_OCR.pdf":("D7", "Permendagri 8/2024 - PKB & BBN-KB 2024"),
    "Permendagri Nomor 6 Tahun 2023.pdf":("D8",  "Permendagri 6/2023 - PKB & BBN-KB 2023"),
    "2024pmkeuangan008.pdf":             ("D9",  "PMK 8/2024 - PPN Kendaraan Listrik (EV)"),
    "5~PMK.010~2022Per.pdf":             ("D10", "PMK 5/2022 - PPnBM Kendaraan Bermotor"),
}

In [4]:
def ekstrak_teks(path: Path) -> str:
    """Ekstrak seluruh teks PDF menjadi satu string (pdfplumber)."""
    bagian = []
    with pdfplumber.open(path) as pdf:
        for halaman in pdf.pages:
            bagian.append(halaman.extract_text() or "")
    return "\n".join(bagian)


def bersihkan(teks: str) -> str:
    """Normalisasi whitespace dan gabungkan kata terpotong tanda hubung."""
    teks = re.sub(r"-\s*\n\s*", "", teks)   # kata terpenggal di akhir baris
    teks = re.sub(r"\s*\n\s*", " ", teks)    # newline -> spasi
    teks = re.sub(r"\s{2,}", " ", teks)        # rapatkan spasi ganda
    return teks.strip()


def pisah_kalimat(teks: str):
    potongan = re.split(r"(?<=[.;])\s+(?=[A-Z(0-9])", teks)
    return [p.strip() for p in potongan if p.strip()]


def _pecah_kata(teks: str, maks=MAX_KATA_CHUNK):
    kata = teks.split()
    return [" ".join(kata[i:i + maks]) for i in range(0, len(kata), maks)]


def chunk_dokumen(teks: str):
    """Pecah dokumen menjadi passage <= KALIMAT_PER_CHUNK kalimat dan <= MAX_KATA_CHUNK kata."""
    kalimat = pisah_kalimat(teks)
    chunks, buf, buf_kata = [], [], 0

    def flush():
        if buf:
            c = " ".join(buf).strip()
            if len(c) >= MIN_CHARS:
                chunks.append(c)

    for kal in kalimat:
        nk = len(kal.split())
        if nk > MAX_KATA_CHUNK:                       # kalimat raksasa (tabel) -> pecah paksa
            flush(); buf, buf_kata = [], 0
            chunks.extend(s for s in _pecah_kata(kal) if len(s) >= MIN_CHARS)
            continue
        if len(buf) >= KALIMAT_PER_CHUNK or buf_kata + nk > MAX_KATA_CHUNK:
            flush(); buf, buf_kata = [], 0
        buf.append(kal); buf_kata += nk
    flush()
    return chunks


def rasio_kata(teks: str) -> float:
    """Proporsi token yang merupakan kata (huruf), bukan angka/kode tabel tarif."""
    tok = teks.split()
    if not tok:
        return 0.0
    wordy = [t for t in tok if sum(c.isalpha() for c in t) >= 0.6 * len(t) and len(t) >= 3]
    return len(wordy) / len(tok)

In [5]:
# Bangun korpus passage berlabel
korpus = []   # list of dict: {chunk_id, doc_id, sumber, teks}
for nama_file, (doc_id, sumber) in PETA_DOKUMEN.items():
    teks = bersihkan(ekstrak_teks(KORPUS_DIR / nama_file))
    # buang passage noise (didominasi angka/kode, mis. baris tabel tarif)
    passages = [p for p in chunk_dokumen(teks) if rasio_kata(p) >= WORDY_MIN]
    for j, passage in enumerate(passages):
        korpus.append({
            "chunk_id": f"{doc_id}#{j}",
            "doc_id":   doc_id,
            "sumber":   sumber,
            "teks":     passage,
        })

print(f"Total passage dalam korpus: {len(korpus)}")

# Ringkasan jumlah passage per dokumen
df_ringkas = (pd.DataFrame(korpus)
              .groupby(["doc_id", "sumber"]).size()
              .reset_index(name="jumlah_passage")
              .sort_values("doc_id", key=lambda s: s.str[1:].astype(int)))
display(df_ringkas)

Total passage dalam korpus: 617


,doc_id,sumber,jumlah_passage
0,D1,UU 12/1985 - PBB,65
2,D2,UU 12/1985 - PBB (salinan),67
3,D3,PMK 85/2024 - Penilaian NJOP PBB-P2,131
4,D4,PMK 234/2022 - PBB pertambangan/kehutanan,56
5,D5,Modul PBB Universitas Terbuka,125
6,D6,Permendagri 7/2025 - PKB & BBN-KB 2025,42
7,D7,Permendagri 8/2024 - PKB & BBN-KB 2024,33
8,D8,Permendagri 6/2023 - PKB & BBN-KB 2023,29
9,D9,PMK 8/2024 - PPN Kendaraan Listrik (EV),38
1,D10,PMK 5/2022 - PPnBM Kendaraan Bermotor,31


In [6]:
# Contoh isi korpus (beberapa passage pertama, panjang bervariasi)
df_korpus = pd.DataFrame(korpus)
df_korpus["jml_kata"] = df_korpus["teks"].str.split().str.len()
display(df_korpus[["chunk_id", "sumber", "jml_kata", "teks"]].head(8))

,chunk_id,sumber,jml_kata,teks
0,D1#0,UU 12/1985 - PBB,160,UNDANG-UNDANG REPUBLIK INDONESIA NOMOR 12 TAHUN 1985 TENTANG PAJAK BUMI DAN BANGUNAN D...
1,D1#1,UU 12/1985 - PBB,42,"kekayaan, telah menimbudkan beban pajak berganda bagi masyarakat, dan oleh karena itu ..."
2,D1#2,UU 12/1985 - PBB,48,"(Lembaran Negara Tahun 1974 Nomor 38, Tambahan Lembaran Negara Nomor 3037); 3. Undang-..."
3,D1#3,UU 12/1985 - PBB,96,"Ordonansi Pajak Rumah Tangga 1908 (Personeele Belasting Ordonnantie 1908, Staatsblad T..."
4,D1#4,UU 12/1985 - PBB,74,"Ordonansi Verponding 1928 (Verpondings Ordonnantie 1928, Staatsblad Tahun 1928 Nomor 3..."
5,D1#5,UU 12/1985 - PBB,80,"Ordonansi Pajak Jalan 1942 (Weggeld Ordonnantie 1942, Staatsblad Tahun 1941 Nomor 97) ..."
6,D1#6,UU 12/1985 - PBB,77,Peraturan Pemerintah Pengganti Undang-undang Nomor 11 Tahun 1959 tentang Pajak Hasil B...
7,D1#7,UU 12/1985 - PBB,63,2. Bangunan adalah konstruksi teknik yang ditanam atau dilekatkan secara tetap pada ta...


## 4. Catatan Preprocessing: TF-IDF vs Transformer

Pada metode TF-IDF/VSM, teks melewati pipeline 5 tahap: *case folding* → *cleaning* →
*tokenizing* → *stopword removal* (Sastrawi) → *stemming* (Sastrawi). Tahap ini cocok untuk
pencocokan **kata kunci**.

Untuk **transformer (IndoBERT) kita TIDAK melakukan stemming/stopword removal.** Model memiliki
*tokenizer subword* sendiri dan memahami **konteks kalimat utuh**; menghapus imbuhan atau kata
fungsi justru merusak makna yang ditangkap model. Jadi passage dikirim ke model dalam bentuk
**teks asli (apa adanya)**. Inilah perbedaan mendasar antara representasi *sparse* (TF-IDF) dan
*dense* (embedding transformer).


## 5. Embedding dengan IndoBERT (Sentence-BERT)

Setiap passage dikodekan menjadi vektor 768 dimensi memakai `firqaaa/indo-sentence-bert-base` —
yaitu **IndoBERT yang di-fine-tune dengan objektif Sentence-BERT**. Versi sentence-embedding ini
menghasilkan vektor yang jauh lebih diskriminatif untuk *semantic search* dibanding IndoBERT mentah
(yang hanya dilatih *masked language modeling*). Embedding dinormalisasi (panjang 1) sehingga
*cosine similarity* setara dengan *dot product*. Pemuatan model pertama kali mengunduh ~500 MB
(tersimpan di `hf_cache/`).


In [7]:
print("Memuat model embedding (unduh ~500MB saat pertama kali)...")
encoder = SentenceTransformer(MODEL_EMBED, device="cpu")
print("Model siap.")


def embed(texts, batch_size=16):
    """Kodekan teks -> matriks (n, 768) ternormalisasi (cosine = dot product)."""
    if isinstance(texts, str):
        texts = [texts]
    return encoder.encode(texts, batch_size=batch_size,
                          normalize_embeddings=True, show_progress_bar=False)

Memuat model embedding (unduh ~500MB saat pertama kali)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model siap.


In [8]:
# Embed seluruh korpus (sekitar 2-4 menit di CPU dengan 4 thread)
teks_korpus = [c["teks"] for c in korpus]
matriks_korpus = embed(teks_korpus)
print("Matriks embedding korpus:", matriks_korpus.shape)   # (jumlah_passage, 768)

Matriks embedding korpus: (617, 768)


## 6. Retrieval — Dense Vector Space Model

Sama seperti VSM klasik, relevansi diukur dengan **cosine similarity**. Bedanya, vektor di sini
bukan bobot TF-IDF melainkan *embedding* IndoBERT yang menangkap makna semantik. Fungsi
`retrieve()` mengembalikan `TOP_K` passage paling relevan terhadap query.


In [9]:
def cosine_sim(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


def retrieve(query: str, top_k: int = TOP_K):
    """Kembalikan top_k passage paling relevan (list dict + skor)."""
    q = embed(query)[0]
    skor = matriks_korpus @ q / (
        np.linalg.norm(matriks_korpus, axis=1) * np.linalg.norm(q) + 1e-9)
    urut = np.argsort(-skor)[:top_k]
    hasil = []
    for rank, idx in enumerate(urut, start=1):
        item = dict(korpus[idx])
        item["skor"] = float(skor[idx])
        item["rank"] = rank
        hasil.append(item)
    return hasil


def tampilkan_retrieval(query: str, top_k: int = TOP_K):
    hasil = retrieve(query, top_k)
    df = pd.DataFrame([{
        "Rank":    h["rank"],
        "Chunk":   h["chunk_id"],
        "Sumber":  h["sumber"],
        "Skor":    round(h["skor"], 4),
        "Cuplikan": h["teks"][:110] + "...",
    } for h in hasil])
    display(df)
    return hasil

In [10]:
# Contoh: retrieval murni hanya MERANKING passage (belum menjawab)
contoh_query = "Berapa tarif Pajak Bumi dan Bangunan?"
print("QUERY:", contoh_query)
_ = tampilkan_retrieval(contoh_query)

QUERY: Berapa tarif Pajak Bumi dan Bangunan?


,Rank,Chunk,Sumber,Skor,Cuplikan
0,1,D5#101,Modul PBB Universitas Terbuka,0.7315,"1.34 Pajak Bumi dan Bangunan ⚫ Bagaimana dengan tarif PBB-nya? Saat ini, tarif PBB ada..."
1,2,D2#49,UU 12/1985 - PBB (salinan),0.7082,"Besarnya Pajak Bumi dan Bangunan yang terhutang : a. Atas tanah = 0,5% x 20% x Rp 240...."
2,3,D5#89,Modul PBB Universitas Terbuka,0.6816,Nilai jual untuk bangunan sebelum diterapkan tarif pajak dikurangi terlebih dahulu den...
3,4,D2#10,UU 12/1985 - PBB (salinan),0.6501,"(3) Batas nilai jual Bangunan Tidak Kena Pajak ditetapkan sebesar Rp. 2.000.000,- (dua..."
4,5,D3#41,PMK 85/2024 - Penilaian NJOP PBB-P2,0.6477,Metode ini terutama diterapkan untuk penentuan NJOP Bumi dan dapat juga digunakan untu...


## 7. Generation — RAG dengan GPT

Passage teratas dirangkai menjadi **konteks** lalu dikirim ke GPT. LLM diminta menjawab
**hanya berdasarkan konteks**, dalam Bahasa Indonesia, dan **mengutip** dokumen sumber `[D#]`.
Jika jawaban tidak ada di konteks, model diinstruksikan untuk menyatakannya secara jujur
(mengurangi halusinasi).


In [11]:
SISTEM_PROMPT = (
    "Anda asisten hukum pajak Indonesia. Jawab pertanyaan HANYA berdasarkan KONTEKS dokumen "
    "yang diberikan. Gunakan Bahasa Indonesia yang jelas dan ringkas. Selalu sertakan sitasi "
    "dokumen sumber dalam format [D#] pada bagian fakta yang relevan. Jika informasi tidak ada "
    "di dalam konteks, katakan dengan jujur bahwa informasi tidak ditemukan pada dokumen. "
    "Jangan mengarang angka atau pasal."
)


def bangun_konteks(passages):
    blok = []
    for p in passages:
        blok.append(f"[{p['doc_id']}] (sumber: {p['sumber']})\n{p['teks']}")
    return "\n\n".join(blok)


def jawab(query: str, top_k: int = TOP_K, tampil=True):
    """Pipeline RAG lengkap: retrieve -> susun konteks -> GPT -> jawaban + referensi."""
    passages = retrieve(query, top_k)
    konteks = bangun_konteks(passages)
    pesan = [
        {"role": "system", "content": SISTEM_PROMPT},
        {"role": "user",
         "content": f"KONTEKS DOKUMEN:\n{konteks}\n\nPERTANYAAN: {query}\n\nJAWABAN:"},
    ]
    resp = client.chat.completions.create(
        model=MODEL_LLM, messages=pesan, temperature=0.2, max_tokens=500)
    jawaban = resp.choices[0].message.content.strip()

    if tampil:
        print("=" * 70)
        print("PERTANYAAN :", query)
        print("=" * 70)
        print(jawaban)
        print("-" * 70)
        print("DOKUMEN REFERENSI (top-{}):".format(top_k))
        for p in passages:
            print(f"  [{p['chunk_id']}] {p['sumber']}  (skor {p['skor']:.3f})")
            print(f"      \"{p['teks'][:120]}...\"")
    return {"query": query, "jawaban": jawaban, "referensi": passages}

## 8. Demo & Perbandingan: VSM (meranking) vs RAG (menjawab)

Untuk pertanyaan yang sama, **retrieval/VSM hanya memberi daftar dokumen** (pengguna harus
membaca sendiri), sedangkan **RAG menghasilkan jawaban langsung** beserta referensinya.


In [12]:
query_demo = "Apa yang dimaksud dengan Kendaraan Bermotor Listrik Berbasis Baterai (KBL)?"

print(">>> (A) RETRIEVAL/VSM — hanya meranking dokumen (pengguna harus membaca sendiri):")
_ = tampilkan_retrieval(query_demo)

print("\n>>> (B) RAG — menjawab pertanyaan + referensi:")
_ = jawab(query_demo)

>>> (A) RETRIEVAL/VSM — hanya meranking dokumen (pengguna harus membaca sendiri):


,Rank,Chunk,Sumber,Skor,Cuplikan
0,1,D9#10,PMK 8/2024 - PPN Kendaraan Listrik (EV),0.7108,7. Kendaraan Bermotor Listrik Berbasis Baterai (Battery Electric Vehicle) yang selanju...
1,2,D7#5,Permendagri 8/2024 - PKB & BBN-KB 2024,0.7073,H 2. Kendaraan Bermotor Listrik Berbasis Baterai (Battery Electric Vehicle) yang selan...
2,3,D8#5,Permendagri 6/2023 - PKB & BBN-KB 2023,0.7027,2. Kendaraan Bermotor Listrik Berbasis Baterai (Battery Electric Vehicle) yang selanju...
3,4,D9#11,PMK 8/2024 - PPN Kendaraan Listrik (EV),0.5645,9. KBL Berbasis ·Baterai Roda Empat Tertentu adalah KBL Berbasis Baterai berupa mobil ...
4,5,D8#19,Permendagri 6/2023 - PKB & BBN-KB 2023,0.5132,(3) Pengenaan PKB dan BBNKB KBL Berbasis Baterai untuk orang atau barang sebagaimana d...



>>> (B) RAG — menjawab pertanyaan + referensi:


PERTANYAAN : Apa yang dimaksud dengan Kendaraan Bermotor Listrik Berbasis Baterai (KBL)?
Kendaraan Bermotor Listrik Berbasis Baterai (KBL Berbasis Baterai) adalah kendaraan yang digerakkan dengan motor listrik dan mendapatkan pasokan sumber daya tenaga listrik dari baterai secara langsung di kendaraan maupun dari luar [D9].
----------------------------------------------------------------------
DOKUMEN REFERENSI (top-5):
  [D9#10] PMK 8/2024 - PPN Kendaraan Listrik (EV)  (skor 0.711)
      "7. Kendaraan Bermotor Listrik Berbasis Baterai (Battery Electric Vehicle) yang selanjutnya disebut KBL Berbasis Baterai ..."
  [D7#5] Permendagri 8/2024 - PKB & BBN-KB 2024  (skor 0.707)
      "H 2. Kendaraan Bermotor Listrik Berbasis Baterai (Battery Electric Vehicle) yang selanjutnya disebut KBL Berbasis Batera..."
  [D8#5] Permendagri 6/2023 - PKB & BBN-KB 2023  (skor 0.703)
      "2. Kendaraan Bermotor Listrik Berbasis Baterai (Battery Electric Vehicle) yang selanjutnya disebut KBL Berbasis Bater

In [13]:
# Beberapa pertanyaan uji lain
pertanyaan_uji = [
    "Apa objek yang dikenakan Pajak Bumi dan Bangunan?",
    "Bagaimana cara penilaian NJOP untuk PBB-P2?",
    "Kendaraan bermotor apa saja yang dikenai PPnBM?",
]
for q in pertanyaan_uji:
    jawab(q)
    print("\n")

PERTANYAAN : Apa objek yang dikenakan Pajak Bumi dan Bangunan?
Objek yang dikenakan Pajak Bumi dan Bangunan (PBB) terdiri atas bumi dan/atau bangunan yang dimiliki, dikuasai, dan/atau dimanfaatkan oleh orang pribadi atau badan. Objek PBB-P2 terdiri atas objek pajak umum dan objek pajak khusus. Objek pajak umum lebih lanjut dibagi menjadi objek pajak standar dan objek pajak nonstandar, yang memiliki konstruksi umum dengan keluasan tanah berdasarkan kriteria tertentu [D3].
----------------------------------------------------------------------
DOKUMEN REFERENSI (top-5):
  [D3#3] PMK 85/2024 - Penilaian NJOP PBB-P2  (skor 0.693)
      "MEMUTUSKAN: Menetapkan : PERATURAN MENTERI KEUANGAN TENTANG PENILAIAN PAJAK BUMI DAN BANGUNAN PERDESAAN DAN PERKOTAAN. P..."
  [D5#33] Modul PBB Universitas Terbuka  (skor 0.688)
      "D. SUBJEK PBB Yang dimaksud subjek pajak bumi dan bangunan adalah orang pribadi atau badan yang secara nyata mempunyai s..."
  [D5#49] Modul PBB Universitas Terbuka  (skor 0.

PERTANYAAN : Bagaimana cara penilaian NJOP untuk PBB-P2?
Cara penilaian NJOP untuk PBB-P2 dilakukan melalui proses Penilaian PBB-P2 yang mencakup beberapa tahapan. Penilaian ini menggunakan beberapa metode, yaitu:

1. Metode perbandingan harga dengan objek lain yang sejenis.
2. Metode nilai perolehan baru.
3. Metode nilai jual pengganti.

Selain itu, penilaian dapat dilakukan secara massal atau individual. Penilaian massal dilakukan secara sistematis untuk sejumlah objek pajak pada saat tertentu dengan menggunakan prosedur standar, seperti Computer Assisted Valuation (CAV) atau Computer Assisted for Mass Appraisal (CAMA) [D3]. 

Dasar pengenaan PBB-P2 adalah NJOP yang ditetapkan berdasarkan proses penilaian tersebut, dan NJOP hasil penilaian dibedakan menjadi NJOP Bumi dan NJOP Bangunan [D3].
----------------------------------------------------------------------
DOKUMEN REFERENSI (top-5):
  [D3#43] PMK 85/2024 - Penilaian NJOP PBB-P2  (skor 0.709)
      "III. TATA CARA PELAKSANAAN PENI

PERTANYAAN : Kendaraan bermotor apa saja yang dikenai PPnBM?
Informasi mengenai kendaraan bermotor yang dikenai PPnBM tidak ditemukan dalam dokumen yang diberikan. Namun, disebutkan bahwa atas penyerahan Barang Kena Pajak yang tergolong mewah berupa kendaraan bermotor tertentu dikenai PPnBM sesuai dengan ketentuan peraturan perundang-undangan [D10]. Untuk rincian lebih lanjut mengenai jenis kendaraan bermotor yang termasuk dalam kategori ini, informasi lebih spesifik tidak tersedia dalam dokumen.
----------------------------------------------------------------------
DOKUMEN REFERENSI (top-5):
  [D6#5] Permendagri 7/2025 - PKB & BBN-KB 2025  (skor 0.664)
      "3. Pajak Kendaraan Bermotor yang selanjutnya disingkat PKB adalah Pajak atas kepemilikan dan/atau penguasaan Kendaraan B..."
  [D10#14] PMK 5/2022 - PPnBM Kendaraan Bermotor  (skor 0.636)
      "663% (enam puluh enam dua per tiga persen) dari PPnBM yang terutang untuk Masa Pajak April 2022 sampai dengan Masa Pajak..."
  [D10#18] 

## 9. Pembahasan

- **Dari meranking ke menjawab.** TF-IDF/VSM mengembalikan daftar dokumen terurut; pengguna
  tetap harus membaca untuk menemukan jawaban. RAG menambahkan LLM yang **membaca konteks dan
  menyusun jawaban** dalam bahasa alami, lengkap dengan sitasi `[D#]`.
- **IndoBERT vs TF-IDF.** TF-IDF mencocokkan kata secara *literal* (sparse). IndoBERT
  menangkap **makna semantik** (dense), sehingga query yang tidak memakai kata persis sama dengan
  dokumen tetap dapat ditemukan (mis. "mobil listrik" ≈ "kendaraan bermotor listrik").
- **Sentence-BERT penting untuk retrieval.** IndoBERT mentah (hanya *masked language modeling*) +
  *mean pooling* menghasilkan embedding yang kurang diskriminatif — pada percobaan, query tentang
  PKB/kendaraan listrik justru tertarik ke dokumen PBB. Memakai IndoBERT yang **di-fine-tune
  Sentence-BERT** (`firqaaa/indo-sentence-bert-base`) membuat passage yang benar-benar relevan
  naik ke peringkat atas.
- **Grounding & sitasi** menekan **halusinasi**: LLM diinstruksikan menjawab hanya dari konteks
  yang diambil retriever, dan menyatakan jujur bila informasi tidak ada.
- **Keterbatasan pada query komparatif.** Pertanyaan seperti *"dasar pengenaan PKB tahun 2025
  dibanding 2023"* bisa dijawab "tidak ditemukan" walau datanya ada: token tahun ("2025"/"2023")
  menyetir embedding ke D5 (Modul PBB) yang padat angka tahun, dan pertanyaan komparatif menuntut
  dua dokumen sekaligus yang sulit ditangani dense retrieval satu-vektor. Query tanpa token tahun
  (*"dasar pengenaan PKB"*) justru menemukan D6/D7/D8 dengan tepat — jadi ini keterbatasan
  retrieval, bukan bug.
- **Opsi perbaikan (lanjutan, belum diimplementasikan).** Hybrid **BM25 + dense** (menangkap
  singkatan/tahun secara persis sekaligus makna), *query expansion* singkatan, filter metadata
  tahun, atau dekomposisi pertanyaan komparatif menjadi sub-query.
- **Keterbatasan lain.** Kualitas jawaban bergantung pada chunking & retrieval (jika passage
  relevan tidak masuk top-k, jawaban bisa kurang lengkap), serta pada batas token IndoBERT
  (passage panjang dipotong). Tabel tarif yang dipecah paksa juga bisa kehilangan struktur.
- **Privasi & biaya.** Pemanggilan LLM memakai API berbayar (OpenRouter); kunci API disimpan di
  `.env` yang tidak di-commit.
